# Numbers all the way down

This notebook is about a big idea behind language models: **inside the model, text becomes numbers**.

Eric's earlier version of this notebook told the story with model files on a server and Python tools that can open up a model directly. In the browser we have a different set of tools, so this version focuses on what we can still see clearly with WebLLM: **token counts**, **chunking**, and **embeddings**.

An **embedding** is a list of numbers that tries to capture the meaning of a piece of text.

## Step 1: Get set up

Small cells that bring in the Python tools we need, check that your browser can run models, load the WebLLM library, and set up the model loader.

### 1a: Import tools

- `json` turns Python data into text and back. We use it to hand data to JavaScript.
- `math` has square roots and other math functions.
- `pandas` (as `pd`) makes and filters tables.
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.
- `display` lets a cell show something and then update it in place (we use it for a progress bar).

In [ ]:
import json
import math
import pandas as pd
import pyjs
from IPython.display import display

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Check for WebGPU

Models run on your computer's graphics chip (GPU). Browsers reach the GPU through a feature called **WebGPU**. Recent Chrome and Edge on a laptop have it; many phones and older browsers do not.

In [ ]:
if getattr(js.navigator, "gpu", None) is not None:
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

### 1c: Load WebLLM

**WebLLM** is a free JavaScript library that runs language models inside the browser. This line fetches it from the internet (it takes a few seconds the first time).

In [ ]:
webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

### 1d: Translate between Python and JavaScript

WebLLM runs in JavaScript, and JavaScript cannot read Python dictionaries directly. Both languages understand **JSON**, a plain-text format for data, so we translate by way of JSON text:

- `js_obj()` turns a Python dictionary into a JavaScript object, for requests we send to WebLLM.
- `py_obj()` goes the other way, turning a JavaScript object (like the model's reply) into Python dictionaries and lists.

You met `js_obj()` in Step 7c of the starter notebook.

In [ ]:
def js_obj(d):
    """Turn a Python dict into a JavaScript object (by way of JSON text)."""
    return js.JSON.parse(json.dumps(d))


def py_obj(value):
    """Turn a JavaScript object into Python dicts and lists (by way of JSON text)."""
    return json.loads(js.JSON.stringify(value))


print("Translators ready.")

### 1e: Set up the model loader

This notebook uses **two** models at once: a chat model and an embedding model. Each one runs in its own **engine** (the part of WebLLM that actually runs a model).

- `chat_engine` and `embed_engine` start as `None` (Python for "nothing yet").
- `progress_bar()` shows a bar that fills up while a model downloads or loads.
- `start_model(model_id, engine)` gets a model ready in an engine and hands the engine back. If the model is not saved in your browser yet, it downloads it first.

This is almost the same loader as in the starter notebook. You do not need to follow every line. Just run it.

In [ ]:
chat_engine = None   # runs the chat model
embed_engine = None  # runs the embedding model


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    # JavaScript needs its own handle on the Python function to call it
    js_show, handle = pyjs.create_callable(show)
    return status, js_show, handle


async def start_model(model_id, engine):
    """Load a model into an engine (making the engine if needed) and return the engine."""
    status, progress, handle = progress_bar()

    if engine is None:
        engine = webllm.MLCEngine.new()   # create an empty engine
    engine.setInitProgressCallback(progress)

    try:
        await engine.reload(model_id)
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await engine.reload(model_id, js_obj({"sliding_window_size": -1}))
        else:
            raise
    return engine


print("Model loader is ready.")

## Step 2: Load the chat model

We use one small chat model to count tokens. If it is not saved in your browser yet, it downloads first.

In [ ]:
CHAT_MODEL = "gemma3-1b-it-q4f16_1-MLC"

chat_engine = await start_model(CHAT_MODEL, chat_engine)
print(f"{CHAT_MODEL} is loaded and ready.")

## Step 3: Raw text is still just text to us

Before the model sees text as tokens, we can measure easy things like **characters** (letters, spaces and punctuation) and **bytes**. A byte is a very small unit of storage. Computers store all text as bytes; most English letters take one byte each, but other characters, like `é` or an emoji, take more.

In [ ]:
TEXT = "A tariff is a tax on imported goods that raises their price."

print(TEXT)
print()
print("Characters:", len(TEXT))
print("Bytes in UTF-8:", len(TEXT.encode("utf-8")))

## Step 4: Prompt tokens are the model's input units

A **token** is a chunk of text the model reads as one unit. Tokens are not always words. Some are short word pieces, punctuation marks, or a space attached to the start of a word.

WebLLM does not show us the individual tokens, but every reply reports how many tokens went in (`prompt_tokens`) and came out (`completion_tokens`).

### 4a: Count the tokens in one request

We send the text with `max_tokens` set to `1`, because we only care about the count going in, not the answer. Then we look at the `usage` part of the reply.

In [ ]:
request = {
    "messages": [{"role": "user", "content": TEXT}],
    "max_tokens": 1,
    "temperature": 0.0,
}

reply = py_obj(await chat_engine.chat.completions.create(js_obj(request)))
reply["usage"]

The count is a bit higher than you might guess. That is because the chat format wraps your message in a few special tokens that mark where a turn starts and ends, and those are counted too.

### 4b: A helper for counting

`prompt_tokens()` does the same thing for any text and returns just the count.

In [ ]:
async def prompt_tokens(text):
    """How many tokens the model reads when we send `text` as one user message."""
    request = {
        "messages": [{"role": "user", "content": text}],
        "max_tokens": 1,
        "temperature": 0.0,
    }
    reply = py_obj(await chat_engine.chat.completions.create(js_obj(request)))
    return reply["usage"]["prompt_tokens"]


print(await prompt_tokens(TEXT))

### 4c: Longer text, more tokens

Now count a few texts of different lengths and put them in a table next to their character counts.

In [ ]:
EXAMPLES = [
    "trade",
    "international trade",
    "International trade can raise living standards.",
    "International trade can raise living standards, but it can also create winners and losers.",
]

rows = []
for text in EXAMPLES:
    rows.append({
        "text": text,
        "characters": len(text),
        "prompt_tokens": await prompt_tokens(text),
    })

pd.DataFrame(rows)

## Step 5: Small wording changes can change token counts

Even when two sentences mean almost the same thing, their token counts may differ. Capital letters and punctuation can count too.

In [ ]:
COMPARISONS = [
    "inflation hurts savers",
    "Inflation hurts savers.",
    "Inflation hurts some savers.",
    "Inflation hurts some savers, especially if prices rise quickly.",
]

rows = []
for text in COMPARISONS:
    rows.append({"text": text, "prompt_tokens": await prompt_tokens(text)})

pd.DataFrame(rows)

## Step 6: Output is also counted in tokens

The model writes its answer one token at a time. Here we ask the same question with different output limits (`max_tokens`) and see how many tokens it actually wrote. If it hits the limit, `finish_reason` is `"length"`.

In [ ]:
PROMPT = "Explain comparative advantage in one short paragraph."

rows = []
for limit in [20, 60, 120]:
    request = {
        "messages": [{"role": "user", "content": PROMPT}],
        "max_tokens": limit,
        "temperature": 0.0,
        "frequency_penalty": 0.5,
    }
    reply = py_obj(await chat_engine.chat.completions.create(js_obj(request)))
    text = reply["choices"][0]["message"]["content"]
    rows.append({
        "max_tokens": limit,
        "finish_reason": reply["choices"][0].get("finish_reason"),
        "completion_tokens": reply["usage"]["completion_tokens"],
        "preview": text[:80] + ("..." if len(text) > 80 else ""),
    })

pd.DataFrame(rows)

## Step 7: Long text gets broken into chunks

A **chunk** is a smaller piece cut from a longer text. Chunking matters for search and memory systems (like notebook 04) because you often store or compare one chunk at a time.

`chunk_text()` cuts the text every `CHUNK_SIZE` characters. Each chunk starts `OVERLAP` characters before the previous one ended, so a sentence cut at the edge still appears whole in at least one chunk. Then we count each chunk's tokens.

In [ ]:
CHUNK_SIZE = 120
OVERLAP = 30

PASSAGE = (
    "Comparative advantage says that people or countries should focus on what they can produce at a lower opportunity cost. "
    "Even if one side is better at making everything, trade can still help both sides if they specialize differently. "
    "That is why economists often use comparative advantage to explain the gains from trade."
)


def chunk_text(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks


rows = []
for number, chunk in enumerate(chunk_text(PASSAGE, CHUNK_SIZE, OVERLAP), start=1):
    rows.append({
        "chunk": number,
        "characters": len(chunk),
        "prompt_tokens": await prompt_tokens(chunk),
        "text": chunk,
    })

pd.DataFrame(rows)

## Step 8: Load the embedding model

Now we switch from counting tokens to **embeddings**, which are also numbers. An embedding model does not chat; it turns each piece of text into a list of numbers. It runs in its own engine, so the chat model stays loaded too.

In [ ]:
EMBED_MODEL = "snowflake-arctic-embed-m-q0f32-MLC-b4"

embed_engine = await start_model(EMBED_MODEL, embed_engine)
print(f"{EMBED_MODEL} is loaded and ready.")

## Step 9: Embeddings are vectors of numbers

A **vector** is an ordered list of numbers. The embedding model turns each input text into one vector.

This model expects each text to be wrapped in the markers `[CLS]` (start) and `[SEP]` (end), so we add them. The request goes to `embed_engine.embeddings.create(...)`, and each item in the reply's `data` list holds one `embedding`.

In [ ]:
WORDS = ["inflation", "deflation", "tariff", "trade", "recession"]

request = {"input": ["[CLS] " + word + " [SEP]" for word in WORDS]}
reply = py_obj(await embed_engine.embeddings.create(js_obj(request)))
embeddings = [item["embedding"] for item in reply["data"]]

print(f"The first 12 numbers for '{WORDS[0]}':")
print(embeddings[0][:12])
print()
print("Vector length:", len(embeddings[0]))

## Step 10: Similar meanings should have more similar vectors

We can compare two vectors with **cosine similarity**. Think of each vector as an arrow; cosine similarity measures how closely two arrows point in the same direction. A score near 1 means very similar; lower means less similar.

In [ ]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


pairs = []
for i in range(len(WORDS)):
    for j in range(i + 1, len(WORDS)):
        pairs.append({
            "word_1": WORDS[i],
            "word_2": WORDS[j],
            "similarity": cosine_similarity(embeddings[i], embeddings[j]),
        })

pd.DataFrame(pairs).sort_values("similarity", ascending=False).reset_index(drop=True)

## Step 11: Putting it all together

When you type a prompt, the model does not think in whole sentences. Very roughly, the path looks like this:

1. raw text
2. tokens (each one is really a number)
3. many layers of number-crunching inside the model
4. new output tokens, one at a time
5. text again

For embeddings, the path ends in a vector of numbers instead of a written answer.

## Your turn

Try your own text in Steps 4 to 7 and your own words in Step 9 (then rerun Step 10). Which pairs look most similar? Which sentences get more tokens than you expected?